# 03 — Credit Risk Modeling

## Objective

The objective of this notebook is to develop and evaluate models that
predict whether a credit-card customer will default on their next
payment.

Two models are evaluated:

1. Logistic Regression — used as an interpretable baseline.
2. XGBoost — used as the primary nonlinear tree-based candidate.

The models are evaluated using the same train/validation/test splits.

Model selection will consider more than accuracy because the dataset
contains a meaningful minority class of customers who default.

## Modeling Strategy

The modeling process follows a three-way data split:

- Training set → fit the models
- Validation set → select modeling decisions such as the classification threshold
- Test set → provide the final unbiased evaluation

The test set will not be used to choose model settings or thresholds.

## 1. Load Model-Ready Data

The model uses the feature-engineered dataset produced by
`02_feature_engineering.ipynb`.

Loading this saved dataset makes this notebook independently
reproducible and prevents it from depending on variables created in
earlier notebooks.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

from xgboost import XGBClassifier

In [2]:
feature_path = "../data/processed/credit_risk_features.csv"

df = pd.read_csv(feature_path)

print(f"Dataset shape: {df.shape}")

Dataset shape: (29965, 34)


## 2. Define Features and Target

The target variable is `default`.

The remaining 33 columns are candidate predictors.

The target is not included among the model features because doing so
would cause target leakage.

In [3]:
X = df.drop(columns="default")
y = df["default"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (29965, 33)
Target shape: (29965,)


## 3. Feature Preprocessing Strategy

Gender, education, and marital status are categorical variables even
though they are represented by integer codes.

They will therefore be one-hot encoded.

The remaining variables are treated as numerical/ordered financial
features for this initial modeling stage.

Logistic Regression will use standardization for numerical variables,
while XGBoost will use the numerical variables without scaling because
tree-based models do not require standardization.

In [4]:
categorical_features = [
    "gender",
    "education",
    "marital_status"
]

numerical_features = [
    col for col in X.columns
    if col not in categorical_features
]

print("Categorical features:", categorical_features)
print("Numerical feature count:", len(numerical_features))

Categorical features: ['gender', 'education', 'marital_status']
Numerical feature count: 30


## 4. Train / Validation / Test Split

A three-way split is used to separate model fitting from model
selection and final evaluation.

Stratification is applied because default is the minority class and we
want approximately the same default rate in all three datasets.

The split is:

- 60% training
- 20% validation
- 20% testing

In [5]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.25,
    random_state=42,
    stratify=y_train_val
)

In [6]:
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nDefault rates:")
print("Train:", round(y_train.mean(), 4))
print("Validation:", round(y_val.mean(), 4))
print("Test:", round(y_test.mean(), 4))

Training: (17979, 33)
Validation: (5993, 33)
Test: (5993, 33)

Default rates:
Train: 0.2213
Validation: 0.2213
Test: 0.2213


## 5. Baseline Model — Logistic Regression

Logistic Regression is used as the baseline because it is relatively
simple and interpretable.

A baseline gives us a reference point. A more complex model should only
be preferred if it provides meaningful improvement over this reference.

Class weighting is used because default is the minority class. This
gives defaulting customers greater influence during model fitting.

In [7]:
logistic_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

In [8]:
logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", logistic_preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

In [9]:
logistic_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](33,)","['credit_limit','gender','education',...,'total_payment_6m', 'recent_payment_3m','older_payment_3m']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,33
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``re

In [10]:
# Generate validation probabilities:
lr_val_proba = logistic_pipeline.predict_proba(X_val)[:, 1]

## 6. Candidate Model — XGBoost

XGBoost is evaluated because credit default risk can involve nonlinear
relationships and interactions between repayment behavior, credit
utilization, demographic characteristics, and payment amounts.

Unlike Logistic Regression, XGBoost does not require numerical
standardization.

The initial configuration is intentionally moderate rather than
heavily tuned. We first want to establish whether the model provides
meaningful improvement over the baseline.

In [11]:
xgb_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            "passthrough",
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

In [12]:
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

In [13]:
xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", xgb_preprocessor),
        ("model", xgb_model)
    ]
)

In [14]:
xgb_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](33,)","['credit_limit','gender','education',...,'total_payment_6m', 'recent_payment_3m','older_payment_3m']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,33
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``re

In [15]:
xgb_val_proba = xgb_pipeline.predict_proba(X_val)[:, 1]

## 7. Model Comparison

Accuracy alone is not sufficient for this problem because the target
is imbalanced and missing a true default can be more important than
simply maximizing the number of correct predictions.

Therefore, model performance is compared using:

- Precision
- Recall
- F1-score
- ROC-AUC
- PR-AUC

ROC-AUC evaluates overall ranking ability, while PR-AUC is particularly
useful when the positive class is less common.

In [16]:
# Create predictions at the default 0.50 threshold:
lr_val_pred = (lr_val_proba >= 0.50).astype(int)
xgb_val_pred = (xgb_val_proba >= 0.50).astype(int)

In [17]:
def evaluate_predictions(name, y_true, y_pred, y_proba):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_proba),
        "PR-AUC": average_precision_score(y_true, y_proba)
    }

In [18]:
comparison = pd.DataFrame([
    evaluate_predictions(
        "Logistic Regression",
        y_val,
        lr_val_pred,
        lr_val_proba
    ),
    evaluate_predictions(
        "XGBoost",
        y_val,
        xgb_val_pred,
        xgb_val_proba
    )
])

comparison.round(4)

,Model,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Logistic Regression,0.7510,0.4544,0.6237,0.5257,0.7763,0.5410
1,XGBoost,0.8186,0.6626,0.3673,0.4726,0.7850,0.5629


### Model Comparison — Interpretation

Logistic Regression provides the stronger recall and F1-score at the
default 0.50 classification threshold, meaning it identifies more of
the customers who default.

XGBoost provides substantially higher accuracy and precision and also
achieves higher ROC-AUC and PR-AUC. The higher ROC-AUC and PR-AUC indicate
that XGBoost has better overall ability to rank customers according to
default risk.

Because our platform is intended to estimate risk and probability of
default rather than rely exclusively on a fixed 0.50 classification
threshold, XGBoost is selected as the candidate model for the next stage.

The lower XGBoost recall at the default threshold is addressed separately
through validation-based threshold selection rather than by selecting a
model solely from its 0.50 classification results.

## 8. Classification Threshold Selection

The default classification threshold of 0.50 is not automatically the
best threshold for a credit-risk application.

A lower threshold can identify more potential defaults, increasing
recall, but it also produces more false positives.

The threshold will therefore be selected using the validation set.
The final test set remains untouched until the selected threshold has
been fixed.

In [19]:
threshold_results = []

for threshold in np.arange(0.10, 0.71, 0.05):
    pred = (xgb_val_proba >= threshold).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_val, pred, zero_division=0
        ),
        "recall": recall_score(
            y_val, pred, zero_division=0
        ),
        "f1": f1_score(
            y_val, pred, zero_division=0
        )
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df.round(4)

,threshold,precision,recall,f1
0,0.10,0.3064,0.8793,0.4545
1,0.15,0.3714,0.7790,0.5030
2,0.20,0.4270,0.6727,0.5224
3,0.25,0.4951,0.6078,0.5457
4,0.30,0.5455,0.5475,0.5465
5,0.35,0.5823,0.4910,0.5327
6,0.40,0.6182,0.4457,0.5180
7,0.45,0.6411,0.4027,0.4947
8,0.50,0.6626,0.3673,0.4726
9,0.55,0.6934,0.3258,0.4433


In [20]:
best_f1_row = threshold_df.loc[
    threshold_df["f1"].idxmax()
]

selected_threshold = float(best_f1_row["threshold"])

print(best_f1_row)
print(f"\nSelected threshold: {selected_threshold:.2f}")

threshold    0.300000
precision    0.545455
recall       0.547511
f1           0.546481
Name: 4, dtype: float64

Selected threshold: 0.30


### Threshold Selection — Interpretation

The default classification threshold of 0.50 produced relatively high
precision but lower recall for the default class.

The validation set was therefore used to evaluate alternative thresholds.
A threshold of 0.30 produced the highest F1-score among the tested
thresholds, providing a more balanced trade-off between precision and
recall.

The threshold of 0.30 is therefore selected as the project classification
threshold.

This threshold is a project-level decision rule and is separate from the
model's predicted probability. It was selected using validation data,
while the test set remained untouched.

## 9. Final Test Evaluation

The test set has not been used to select the model or classification
threshold.

The selected XGBoost threshold from the validation set is now applied
once to the test predictions to obtain the final performance estimate.

In [21]:
xgb_test_proba = xgb_pipeline.predict_proba(X_test)[:, 1]

xgb_test_pred = (
    xgb_test_proba >= selected_threshold
).astype(int)

In [22]:
final_metrics = evaluate_predictions(
    "XGBoost",
    y_test,
    xgb_test_pred,
    xgb_test_proba
)

pd.Series(final_metrics)

Model         XGBoost
Accuracy     0.790923
Precision    0.527097
Recall       0.535445
F1           0.531238
ROC-AUC      0.772972
PR-AUC       0.550977
dtype: object

In [23]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, xgb_test_pred))

Confusion Matrix:
[[4030  637]
 [ 616  710]]


### Final Test Evaluation — Interpretation

The final XGBoost model was evaluated on the untouched test set using
the classification threshold selected from the validation set.

The model achieved a ROC-AUC of 0.7730 and PR-AUC of 0.5510, indicating
useful discrimination between default and non-default customers.

At the selected threshold of 0.30, the model achieved 53.54% recall and
52.71% precision for the default class.

The confusion matrix shows that 710 of 1,326 actual default cases were
identified, while 616 were missed. The model also produced 637 false
positive predictions.

These results indicate that the model provides meaningful credit-risk
ranking and screening capability, but it should not be interpreted as a
perfect decision-maker. The model's predicted probabilities will be
examined separately in the Probability of Default and Risk Segmentation
stage.

## 10. Modeling Conclusion

Logistic Regression was established as the interpretable baseline.

XGBoost was selected as the candidate credit-risk model because it
provided stronger ROC-AUC and PR-AUC performance and better precision
at the default classification threshold.

The default threshold of 0.50 was not retained because it produced low
recall for the default class. Validation-based threshold analysis
selected a threshold of 0.30, which provided the highest F1-score among
the tested thresholds.

On the untouched test set, the final XGBoost configuration achieved:

- ROC-AUC: 0.7730
- PR-AUC: 0.5510
- Recall: 0.5354
- Precision: 0.5271
- F1-score: 0.5312

The model is therefore retained for the next stage, where its predicted
probabilities will be examined as model-estimated Probability of Default
(PD) and converted into project-level risk categories.